In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from src.imputation import imputation_normal_distribution, log2
from src.data_processing import pre_processing, calculate_data_completeness, filter_data, summarize_filtered_data
from src.data_processing import compute_pca,generate_pca_plot, calculate_cv
from src.utils import lookup
from src.statistical_testing import perform_linear_regression
import pingouin as pg
from matplotlib_venn import venn3, venn3_circles
from venn import venn
import statsmodels.stats.multitest as multi
from tqdm import tqdm
from scipy.stats import pearsonr
from scipy.stats import zscore
import pickle
import string
import os
from pathlib import Path

In [ ]:
import os
from pathlib import Path

# Get the number of available CPUs
CPUS = os.cpu_count()

# Define the paths for the raw and processed data folders
DATA_FOLDER_2k = '/Volumes/auditgroupdirs/SUND-CPR-TARGET_PROTEOMICS/2k_discovery'
DATA_FOLDER_RAW = Path(os.path.join(DATA_FOLDER_2k, 'data/raw'))
DATA_FOLDER_PROCESSED = Path(os.path.join(DATA_FOLDER_2k, 'data/processed'))
DATA_FOLDER_CLINIC = '/Volumes/auditgroupdirs/SUND-CBMR-Childhood-Genetic-TCOC/Proteomics analysis/GitHub/TARGET/'

# Ensure base folders are created and define subfolder paths
os.makedirs(DATA_FOLDER_PROCESSED, exist_ok=True)
subfolders = ['tables', 'results', 'figures', 'pQTL', 'dash', 'annotations', 'GWAS']
folders = {f: Path(DATA_FOLDER_2k, f) for f in subfolders}

# Create subfolders if they don't exist
for folder in folders.values():
    folder.mkdir(parents=True, exist_ok=True)
    
Path(folders['pQTL'] / 'gemma').mkdir(exist_ok=True)
gemma_path = Path(folders['pQTL'], 'gemma')
pep_dir = Path(os.path.join(folders['pQTL'], 'peptide_validation'))

### Read annotation file

In [ ]:
# Read annotation file
annotation_file = pd.read_csv(os.path.join(DATA_FOLDER_RAW, 'annotations.csv'), sep=';')
IDmapping_sampleID_to_Batch = dict(zip(annotation_file['Sample ID'], annotation_file['Grouping_batch']))
IDmapping_sampleID_to_Instrument = dict(zip(annotation_file['Sample ID'], annotation_file['Instrument']))

### Read peptide export

In [ ]:
pep_file = '/Users/jpx667/Downloads/20240210_214710_Peptide Lili long (Normal).tsv'
pep_file = '/Users/jpx667/Downloads/20240225_162254_Peptide Lili long (Normal).tsv'
pep_file = '/Users/jpx667/Downloads/20240229_010223_Peptide Lili long (Normal)_canonical_specific.tsv'
cols_to_keep = ['R.FileName', 'PG.Genes', 'PG.ProteinAccessions', 'PEP.AllOccurringProteinAccessions',
                'PEP.PeptidePosition','PEP.StrippedSequence','PEP.Quantity', 'PEP.IsProteinGroupSpecific',
                'PEP.UsedForProteinGroupQuantity','EG.ModifiedSequence', ]
output_file = '/Users/jpx667/Documents/HOLBAEK/data_raw_pep_long.pkl'
RE_READ = False
if not RE_READ:
    with open(output_file, 'rb') as handle:
        data_raw_pep_long = pickle.load(handle)
else:
    chunk_size = 10000
    file_to_read = pd.read_csv(pep_file, sep='\t', usecols=cols_to_keep, chunksize=chunk_size, na_values='Filtered')

    chunks = []
    for chunk in tqdm(file_to_read):
        chunks.append(chunk)
    report_peptide = pd.concat(chunks)
    data_raw_pep_long = report_peptide.dropna().reset_index().drop(['index'], axis=1)
    with open(output_file, 'wb') as pfile:
        pickle.dump(data_raw_pep_long, pfile, protocol=pickle.HIGHEST_PROTOCOL)

#### Preprocess data

In [ ]:
RE_PROCESS = False
output_file = '/Users/jpx667/Documents/HOLBAEK/data_raw_pep_long_processed.pkl'
if not RE_PROCESS:
    with open(output_file, 'rb') as handle:
        data_raw_long = pickle.load(handle)
else:
    data_raw_long = pre_processing(data_raw_pep_long)
    data_raw_long['PeptideID']=data_raw_long['ProteinID_Genename']+'_'+data_raw_long['PEP.StrippedSequence']
    data_raw_long=data_raw_long.drop_duplicates().dropna()
    print(data_raw_long.shape)
    with open(output_file, 'wb') as pfile:
        pickle.dump(data_raw_long, pfile, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
peptide_ids = data_raw_long[['Gene names', 'Protein IDs','PEP.AllOccurringProteinAccessions',
                             'Gene name', 'Protein ID','PEP.IsProteinGroupSpecific',
                             'ProteinID_Genename', 'PEP.StrippedSequence', 'PEP.PeptidePosition',
                             'PEP.UsedForProteinGroupQuantity', 'PeptideID']].drop_duplicates()

In [ ]:
peptide_ids['PEP.IsProteinGroupSpecific'].value_counts()

In [ ]:
cols_to_keep = ['Sample ID', 'PeptideID', 'PEP.Quantity']
data_peptide_raw = data_raw_long.reset_index()[cols_to_keep].drop_duplicates().pivot(columns='Sample ID', index='PeptideID', values='PEP.Quantity')

In [ ]:
df_raw_comp = calculate_data_completeness(data_peptide_raw)
sns.lineplot(x='rank', y='%Complete', data=df_raw_comp)

In [ ]:
peptides, sample_ids, data_peptide_filtered = filter_data(data_peptide_raw, protein_wise_thresh_perc=0.4)

In [ ]:
summarize_filtered_data(data_peptide_filtered)

In [ ]:
nr_peptide_per_sample = data_raw_long[['Protein IDs', 'Gene names','PEP.StrippedSequence']].drop_duplicates().groupby('Protein IDs')['PEP.StrippedSequence'].count()

In [ ]:
pd.DataFrame(nr_peptide_per_sample).to_csv(os.path.join(DATA_FOLDER_PROCESSED, 'nr_peptide_per_protein.csv'))

#### Plot peptides by data completeness

In [ ]:
df_filtered_comp = calculate_data_completeness(data_peptide_filtered)
sns.lineplot(x='rank', y='%Complete', data=df_filtered_comp)
plt.ylim(0, 1.1)

In [ ]:
data_peptide_filtered_log = data_peptide_filtered.apply(log2)

#### PCA

In [ ]:
X_train = data_peptide_filtered_log.T.dropna(axis=1)
pca, df_pc, df_loadings = compute_pca(dataframe=X_train)
#df_loadings['Gene name'] = df_loadings.index.str.split('_').str[1]
df_pc = df_pc.join(annotation_file.set_index('Sample ID'), how='left')
fig_pca = generate_pca_plot(df_pc=df_pc, pca=pca, PCA_x=1, PCA_y=2)
#fig_pca.savefig(os.path.join(folders['figures'], 'PCA.pdf'), bbox_inches='tight', dpi=120)

#### Imputation

In [ ]:
data_pep_filtered_imputed = data_peptide_filtered_log.apply(imputation_normal_distribution)

#### Normalization

In [ ]:
from combat.pycombat import pycombat
dm = data_pep_filtered_imputed[sample_ids].copy()

batch_sample_prep = [IDmapping_sampleID_to_Batch[i] for i in dm.columns]
batch_instrument = [IDmapping_sampleID_to_Instrument[i] for i in dm.columns]
dm1 = pycombat(dm, batch_sample_prep)
dm2 = pycombat(dm1, batch_instrument)

#### Combine clinical data

In [ ]:
data_cli_prot = pd.read_csv(folders['pQTL'] / 'phenomics/data_cli_prot.csv').set_index('Sample ID')

#### Data after normalization with no imputed values

In [ ]:
mask = data_peptide_filtered.isna()
data_reverted = dm2.mask(mask)
data_combined_noimpute = data_reverted.T.join(data_cli_prot).rename_axis('Sample ID', axis=0)
data_combined_noimpute = data_combined_noimpute.join(annotation_file.set_index('Sample ID'), how='left')

In [ ]:
IDmapping_SampleID_to_bloodSampleID = pd.read_pickle(os.path.join(folders['pQTL'], 'phenomics', 'IDmapping_SampleID_to_bloodSampleID.p'))

In [ ]:
annotation_file['Participant ID']='66-' + annotation_file['Sample ID'].map(IDmapping_SampleID_to_bloodSampleID)

In [ ]:
peptide_ids['gwas']=np.where(peptide_ids['PeptideID'].isin(peptides), True, False)

In [ ]:
data_pep_export = {'data_noimpute':data_reverted, 'data_cli':data_cli_prot, 
                   'annotation_file':annotation_file, 'peptide_ids':peptide_ids}

In [ ]:
with open(pep_dir / 'data_pep_export.pkl', 'wb') as handle:
    pickle.dump(data_pep_export, handle, protocol=pickle.HIGHEST_PROTOCOL)

#### PCA

In [ ]:
X_train = dm2.T.dropna(axis=1)
pca, df_pc, df_loadings = compute_pca(dataframe=X_train)
#df_loadings['Gene name'] = df_loadings.index.str.split('_').str[1]
df_pc = df_pc.join(annotation_file.set_index('Sample ID'), how='left')
df_pc = df_pc.join(data_cli_prot)
fig_pca = generate_pca_plot(df_pc=df_pc, pca=pca, PCA_x=1, PCA_y=2)
#fig_pca.savefig(os.path.join(folders['figures'], 'PCA.pdf'), bbox_inches='tight', dpi=120)

In [ ]:
fig, ax=plt.subplots(figsize=(4,4))

sns.scatterplot(x='PC1', y='PC2', data=df_pc, hue='time_to_analysis', palette='bwr')

#### INT transformation

In [ ]:
peptides=dm2.index

In [ ]:
# Perform ranked-based inverse normalized transformation (INT) on protein levels per protein
# Refer to https://github.com/edm1/rank-based-INT

from src.rank_based_int import rank_INT
RE_INT = False

if not RE_INT:
    dm_int = pd.read_pickle(pep_dir / 'peptides_int.pkl')
else:
    new_df = []
    for peptide in tqdm(peptides):
        new_df.append(pd.DataFrame(rank_INT(dm2.T[peptide], stochastic=False), columns=[peptide]))
    dm_int = pd.concat(new_df, axis=1)
    dm_int.rename_axis('Sample ID', axis=0, inplace=True)
    dm_int.to_pickle(pep_dir / 'peptides_int.pkl')

In [ ]:
data_combined_int = dm_int.join(data_cli_prot)
data_combined_int = data_combined_int.join(df_pc['PC1'], how='left')

In [ ]:
# overweight/obesity (BMI SDS >= 1.28)
data_combined_int['overweight'] = np.where(data_combined_int['z_BMI.Nysom']>=1.28, 1, 0)
data_combined_int['overweight*z_BMI.Nysom']=data_combined_int['overweight']*data_combined_int['z_BMI.Nysom']

In [ ]:
covariates_pqtl = ['age', 'sex', 'z_BMI.Nysom', 'overweight*z_BMI.Nysom', 
                   'overweight', 'time_to_analysis', 'PC1']

#### Covariate correction

In [ ]:
RE_LIREG = False

if not RE_LIREG:
    stats_pqtl = pd.read_pickle(pep_dir / 'lireg_statistics.pkl')
    residuals_pqtl = pd.read_pickle(pep_dir / 'lireg_residuals.pkl')
else:
    stats_pqtl, residuals_pqtl = perform_linear_regression(data_combined_int, peptides, covariates_pqtl)
    # Save results
    stats_pqtl.to_pickle(pep_dir / 'lireg_statistics.pkl')
    residuals_pqtl = pd.DataFrame.from_dict(residuals_pqtl).rename_axis('Sample ID', axis=0)
    residuals_pqtl.to_pickle(pep_dir / 'lireg_residuals.pkl')

#### INT on residuals

In [ ]:
RE_INT = False

if not RE_INT:
    data_gwas_int = pd.read_pickle(pep_dir / 'lireg_residuals_int.pkl')
else:
    new_df = []
    data = residuals_pqtl
    for peptide in tqdm(data.columns):
        new_df.append(pd.DataFrame(rank_INT(data[peptide], stochastic=False), columns=[peptide]))
    data_gwas_int = pd.concat(new_df, axis=1)
    data_gwas_int.rename_axis('Sample ID', axis=0, inplace=True)
    with open(pep_dir / 'lireg_residuals_int.pkl', 'wb') as handle:
        pickle.dump(data_gwas_int, handle, protocol=pickle.HIGHEST_PROTOCOL)

#### PCA

In [ ]:
X_train = data_gwas_int.dropna(axis=1)
pca, df_pc, df_loadings = compute_pca(dataframe=X_train)
df_loadings['Gene name'] = df_loadings.index.str.split('_').str[1]
df_pc = df_pc.join(annotation_file.set_index('Sample ID'), how='left')
df_pc = df_pc.join(data_cli_prot[['obesity', 'IgA', 'IgG', 'IgM']])
fig_pca = generate_pca_plot(df_pc=df_pc, pca=pca, PCA_x=1, PCA_y=2, group_column='obesity', palette='bwr')

#### Check how many peptides per protein

In [ ]:
df = pd.DataFrame({'Protein ID':data_gwas_int.columns.str.split('_').str[0],
             'Gene name':data_gwas_int.columns.str.split('_').str[1],
             'Stripped sequence':data_gwas_int.columns.str.split('_').str[2]})
df1 = pd.DataFrame(df['Protein ID'].value_counts())
df1[df1['Protein ID']>1]

#### Prepare the data to fit the GEMMA input format

In [ ]:
# Replace sample ID with participant ID so it's compatible with genotype data
participant_ids = '66-' + data_gwas_int.index.map(IDmapping_SampleID_to_bloodSampleID)
data_gwas_int.insert(0, 'Participant ID', participant_ids)

In [ ]:
# Import the .fam file template
fam_tep = pd.read_csv(folders['pQTL'] / 'QC_PLINK/target5.fam', header=None, sep=' ')
data_gwas_export = fam_tep.set_index(0).join(data_gwas_int.round(5).set_index('Participant ID')).reset_index().drop([5], axis=1)
data_gwas_export = data_gwas_export.replace(np.nan, 'NA')

In [ ]:
# Export data for GWAS
data_gwas_export.to_csv(pep_dir / 'export/peptide.fam', header=None, index=False, sep=' ')

In [ ]:
# Create a dataframe mapping phenotype IDs to protein IDs
nr_peptides = len(peptides)
peptideID_pqtl=pd.DataFrame({'Phenotype ID':np.arange(1, nr_peptides+1),'Protein ID':data_gwas_export.columns[5:]})
peptideID_pqtl.to_csv(pep_dir / 'export/PeptideID.txt', index=False, sep='\t')

with open(pep_dir / 'export/phenotype_pep.list', 'w') as file:
    for line in np.arange(1, nr_peptides+1):
        file.write(str(line) + '\n')
        
# Association performed in Computerome (GEMMA v0.98.3)